# Submission Details
- **Full Name:** Chakradhar Reddy Yerragudi
- **Uplevel Email Address:** charadharvasurama@gmail.com
- **Problem Statement:** IMDB Movie Chatbot


## Environment Setup
Install the packages needed for the notebook workflow.


In [ ]:
# Installing the LangChain Hub package to access and manage pre-built AI chains, prompts, and agents.
%pip install langchainhub

# Installing the LangChain OpenAI integration to use OpenAI models within LangChain workflows.
%pip install langchain-openai

# Installing the core LangChain library for building LLM-based applications, including chaining, memory, and retrieval capabilities.
%pip install --upgrade langchain

# Installing the community version of LangChain, which includes integrations and tools contributed by the community.
%pip install langchain-community

# Installing FAISS (Facebook AI Similarity Search) for efficient similarity-based search on text embeddings.
%pip install faiss-cpu

# Installing Gradio, a framework to create web-based UIs for AI models and applications easily.
%pip install gradio

# Installing langchain-text-splitters for text chunking
%pip install langchain-text-splitters


## OpenAI Client Import
Import the OpenAI client used for the early API example.


In [1]:
# Importing the OpenAI library to interact with OpenAI's API services.
from openai import OpenAI

# Import basic libraries
import os
import getpass
import pandas as pd
import numpy as np

## API Key Setup
Load the API key from the environment or prompt for it once.


In [6]:
# Use an existing OpenAI API key when available; otherwise prompt once for it.
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    api_key = getpass.getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key


## Sample Prompt
Define a simple prompt for the initial OpenAI API test.


In [7]:
prompt = ''' What was uber's revenue in 2022? '''
client = OpenAI()
openai_response = client.chat.completions.create(
    model='gpt-3.5-turbo',  # Specifying the model to use;
    # Note: An older model chosen for testing purposes because the cutoff is 2021 whereas prompt is querying details about 2022
    messages=[{'role': 'user', 'content': prompt}]  # Creating a structured message for the AI model
)

## Sample Response Check
Display the response from the initial API example.


In [8]:
openai_response.choices[0].message.content

'I cannot provide that information as it is not publicly available yet.'

## Load Dataset
Read the IMDb dataset into a Pandas dataframe.


In [9]:
# Load the data
df=pd.read_csv('/Users/chakradhar/Documents/Movie Chatbot/IMDb_Dataset.csv')

## Inspect Data
Preview the dataset to understand the available movie fields.


In [10]:
# View & Understand the data
df.head()

,Title,IMDb Rating,Year,Certificates,Genre,Director,Star Cast,MetaScore,Poster-src,Duration (minutes)
0,End of the Spear,6.8,2005,PG-13,Adventure,Jim Hanon,Louie LeonardoChad AllenJack Guzman,45.0,https://m.media-amazon.com/images/M/MV5BMTYxOT...,108.0
1,Elvira Madigan,7.0,1967,PG,Biography,Bo Widerberg,Pia DegermarkThommy BerggrenLennart Malmer,66.0,https://m.media-amazon.com/images/M/MV5BMmY2Nj...,91.0
2,The Kid Stays in the Picture,7.3,2002,R,Documentary,Nanette Burstein,Robert EvansEddie AlbertPeter Bart,75.0,https://m.media-amazon.com/images/M/MV5BZjhiZm...,93.0
3,It Ain't Over,8.2,2022,PG,Documentary,Sean Mullin,Andy AndresRoger AngellMarty Appel,79.0,https://m.media-amazon.com/images/M/MV5BZWViYW...,99.0
4,Mahler,7.0,1974,PG,Biography,Ken Russell,Robert PowellGeorgina HaleLee Montague,66.0,https://m.media-amazon.com/images/M/MV5BYzY4Mz...,115.0


## Build Movie Descriptions
Create a text description for each movie from the dataset columns.


In [11]:
# Create movie description for each movie from the details provided in the dataset
df['movie_description'] = df.apply(
    lambda row: (
        f"{row['Title']} released in {row['Year']} is an {row['Genre']} movie \n"
        f"with an IMDb Rating of {row['IMDb Rating']}. \n"
        f"It is {row['Certificates']} and directed by {row['Director']}, "
        f"starring {row['Star Cast']}. \n"
        f"The duration is {row['Duration (minutes)']} minutes."
    ),
    axis=1
)

## Preview Description
Print one generated movie description as a quick sanity check.


In [12]:
print(df['movie_description'][1])

Elvira Madigan released in 1967 is an Biography movie 
with an IMDb Rating of 7.0. 
It is PG and directed by Bo Widerberg, starring Pia DegermarkThommy BerggrenLennart Malmer. 
The duration is 91.0 minutes.


## Chunk Preparation
Split movie descriptions into chunks for retrieval and search.


In [13]:
# Now, data is ready!
# Its time to create your vector store



# Perform Text Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

movie_metadatas = [
    {
        "row_index": int(idx),
        "title": str(row["Title"]),
    }
    for idx, row in df.iterrows()
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
documents = text_splitter.create_documents(
    df['movie_description'].fillna('').tolist(),
    metadatas=movie_metadatas,
)

## Create Embeddings
Generate embeddings for the text chunks.


In [14]:
# Create embeddings for the chunks
# See https://python.langchain.com/docs/integrations/text_embedding/ for a list of available embedding models on LangChain

from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

## Build Vector Store
Store the embedded chunks in FAISS for fast retrieval.


In [15]:
# Create a vector store using the created chunks and the embeddings model

from langchain_community.vectorstores import FAISS
vector = FAISS.from_documents(documents, embeddings)

## RAG Imports
Import the LangChain components used for the retrieval pipeline.


In [16]:
# Importing ChatOpenAI from LangChain to interact with OpenAI's language models, such as GPT, for generating responses.
from langchain_openai import ChatOpenAI

# Importing ChatPromptTemplate to create structured prompts for the chatbot, ensuring consistent interactions with the AI model.
from langchain_core.prompts import ChatPromptTemplate

# Importing OpenAIEmbeddings to convert text data into numerical vector representations for similarity search and retrieval.
from langchain_openai import OpenAIEmbeddings



from langchain_core.runnables import RunnablePassthrough

# Importing StrOutputParser from LangChain to parse the output
from langchain_core.output_parsers import StrOutputParser

## Create LLM
Initialize the chat model used across the notebook.


In [17]:
# Create the llm model

llm = ChatOpenAI(api_key=os.environ["OPENAI_API_KEY"], model = 'gpt-4o-mini')

## Prompt Template
Create the prompt template for the retrieval-based answer flow.


In [18]:
# Create the prompt template
output_parser = StrOutputParser()
prompt = ChatPromptTemplate.from_template(
    """Answer the following question based only on the provided context:

    <context>
    {context}
    </context>

    Question: {input}""",
    output_parser=output_parser  # The output parser ensures that the response is returned in a structured string format.
)




## Create Retriever
Build a retriever from the vector store.


In [19]:
# Create a retriever from the vector store for fetching relevant documents
# See https://python.langchain.com/v0.1/docs/modules/data_connection/retrievers/vectorstore/

retriever = vector.as_retriever()


## Build RAG Chain
Combine the retriever, prompt, model, and parser into a chain.


In [20]:
# Create the document processing chain/ RAG Chain
# Join retrieved document chunks into one context string for the RAG prompt.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
)

## Invoke RAG Chain
Run the retrieval chain on a sample movie-related question.


In [21]:
# Invoke the retrieval chain to process the user's query
rag_chain.invoke("Best Action movie from 2000 to 2010 with goot rating?")


AIMessage(content='The best Action movie from 2000 to 2010 based on IMDb ratings is "GB: 2525," released in 2009, with a rating of 8.1.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 275, 'total_tokens': 314, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_cd4a20171f', 'id': 'chatcmpl-DRl8khyDsIsLcOHI3N6bkjJLRI42q', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d648c-a9ce-7c43-aac0-ba494b98332e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 275, 'output_tokens': 39, 'total_tokens': 314, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Format RAG Output
Format the chain output so it is easier to read.


In [22]:
# Perform adequate formatting to print the final response in a user readable format
rag_chain.invoke("Best Action movie from 2000 to 2010 with goot rating?").content

'Based on the provided context, the best Action movie from 2000 to 2010 with the highest IMDb rating is "GB: 2525," which was released in 2009 and has an IMDb rating of 8.1.'

## Intermediate UI Test
Launch a simple Gradio check for the earlier retrieval flow.


In [23]:
# Optional: Test the functionality using a Gradio UI (intermediate check)

import gradio as gr

# Send a user query through the simple RAG chain for the intermediate UI demo.
def chatbot_interface(query):
    return rag_chain.invoke(query).content

# Create the Gradio interface
iface = gr.Interface(fn=chatbot_interface,
                     inputs=gr.Textbox(lines=2, placeholder="Enter your query here..."),
                     outputs=gr.Textbox(lines=10, label="Response"), # Changed to gr.Textbox with more lines
                     title="Movie RAG Chatbot",
                     description="Ask questions about movies based on the provided dataset.")

# Launch the interface
iface.launch(debug=True)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


## Define Tools
- `apply_filters` is used for exact filtering with dataset columns like genre, year, rating, actor, and director.
- `semantic_search` is used for fuzzy meaning-based search such as similar themes, tone, or movie style.
- Together, these tools let the chatbot combine structured search with flexible recommendations.


In [24]:
# Define various agents - each performing a particular task using tool decorator

from langchain_core.tools import tool
import json
import re


@tool
# Apply exact dataframe filters such as genre, year, rating, actor, and director.
def apply_filters(query_struct: str) -> str:
    """Apply deterministic Pandas filters for genre, year, rating, actor, and director."""
    match = re.search(r"```json\\s*(.*?)\\s*```", query_struct, re.DOTALL)
    parsed = json.loads(match.group(1) if match else query_struct)
    filters = {
        "genre": parsed.get("genre"),
        "actor": parsed.get("actor"),
        "director": parsed.get("director"),
        "year_min": parsed.get("year_min"),
        "year_max": parsed.get("year_max"),
        "rating_min": parsed.get("rating_min"),
        "semantic_query": parsed.get("semantic_query", "")
    }
    filtered_df = df.copy()

    if filters["genre"]:
        genres = filters["genre"] if isinstance(filters["genre"], list) else [filters["genre"]]
        genre_pattern = "|".join(re.escape(str(genre)) for genre in genres)
        filtered_df = filtered_df[
            filtered_df["Genre"].fillna("").str.contains(genre_pattern, case=False, regex=True)
        ]

    if filters["actor"]:
        filtered_df = filtered_df[
            filtered_df["Star Cast"].fillna("").str.contains(str(filters["actor"]), case=False, regex=False)
        ]

    if filters["director"]:
        filtered_df = filtered_df[
            filtered_df["Director"].fillna("").str.contains(str(filters["director"]), case=False, regex=False)
        ]

    if filters["year_min"] is not None:
        filtered_df = filtered_df[filtered_df["Year"] >= int(filters["year_min"])]

    if filters["year_max"] is not None:
        filtered_df = filtered_df[filtered_df["Year"] <= int(filters["year_max"])]

    if filters["rating_min"] is not None:
        filtered_df = filtered_df[filtered_df["IMDb Rating"] >= float(filters["rating_min"])]

    filtered_df = filtered_df.sort_values(by="IMDb Rating", ascending=False).head(25)
    records = []
    for idx, row in filtered_df.iterrows():
        records.append({
            "row_index": int(idx),
            "title": row["Title"],
            "year": int(row["Year"]) if pd.notna(row["Year"]) else None,
            "genre": row["Genre"],
            "director": row["Director"],
            "actors": row["Star Cast"],
            "imdb_rating": float(row["IMDb Rating"]) if pd.notna(row["IMDb Rating"]) else None,
            "movie_description": row.get("movie_description", ""),
            "similarity_score": 0.0
        })
    return json.dumps(records, indent=2)


@tool
# Find semantically similar movies using the vector store with a text-overlap fallback.
def semantic_search(query_text: str, top_k: int = 10) -> str:
    """Run embedding-based semantic search with FAISS for fuzzy movie queries."""
    best_matches = {}

    try:
        matches = vector.similarity_search_with_score(query_text, k=max(top_k * 3, top_k))

        for doc, raw_score in matches:
            row_index = int(doc.metadata["row_index"])
            similarity_score = 1.0 / (1.0 + float(raw_score))

            if row_index in best_matches and best_matches[row_index]["similarity_score"] >= similarity_score:
                continue

            row = df.iloc[row_index]
            best_matches[row_index] = {
                "row_index": row_index,
                "title": row["Title"],
                "year": int(row["Year"]) if pd.notna(row["Year"]) else None,
                "genre": row["Genre"],
                "director": row["Director"],
                "actors": row["Star Cast"],
                "imdb_rating": float(row["IMDb Rating"]) if pd.notna(row["IMDb Rating"]) else None,
                "movie_description": row.get("movie_description", doc.page_content),
                "similarity_score": similarity_score
            }
    except Exception:
        query_tokens = set(re.findall(r"\w+", query_text.lower()))
        scored_rows = []

        for idx, row in df.iterrows():
            description = str(row.get("movie_description", "")).lower()
            description_tokens = set(re.findall(r"\w+", description))
            overlap = len(query_tokens & description_tokens)
            if overlap == 0:
                continue

            similarity_score = overlap / max(len(query_tokens), 1)
            scored_rows.append((idx, similarity_score))

        for row_index, similarity_score in sorted(scored_rows, key=lambda item: item[1], reverse=True)[:top_k]:
            row = df.iloc[row_index]
            best_matches[row_index] = {
                "row_index": int(row_index),
                "title": row["Title"],
                "year": int(row["Year"]) if pd.notna(row["Year"]) else None,
                "genre": row["Genre"],
                "director": row["Director"],
                "actors": row["Star Cast"],
                "imdb_rating": float(row["IMDb Rating"]) if pd.notna(row["IMDb Rating"]) else None,
                "movie_description": row.get("movie_description", ""),
                "similarity_score": similarity_score
            }

    records = sorted(best_matches.values(), key=lambda item: item["similarity_score"], reverse=True)[:top_k]
    return json.dumps(records, indent=2)


tools = [apply_filters, semantic_search]


## Build Orchestrator
- The orchestrator decides whether a tool is needed for the user query.
- It cleans the tool plan, runs the selected tools, merges results, and ranks them.
- It also keeps session memory so follow-up questions can use earlier conversation context.
- Finally, it generates a natural response for the user.

- `get_memory(session_id)`: stores chat history per session.
- `format_chat_history(history)`: converts recent messages into prompt-ready text.
- `clear_chat_history(session_id)`: clears stored history for one session.
- `extract_json_payload(value)`: safely parses model output into JSON.
- `infer_query_struct(user_query)`: extracts simple filters from the query.
- `build_fallback_plan(user_query)`: creates a backup plan if model routing fails.
- `normalize_tool_plan(model_output, user_query)`: standardizes the tool plan format.
- `remove_reference_movies(user_query, ranked_results)`: avoids recommending the reference movie again.
- `tool_chain(...)`: runs tools, merges outputs, and ranks results.
- `generate_final_response(...)`: writes the final user-facing answer.
- `run_movie_chain(inputs)`: connects the full workflow from query to response.
- `RunnableWithMessageHistory(...)`: enables session-based follow-up memory.


### Orchestrator Setup
- Imports the LangChain components used in routing and memory.
- Builds the system prompt that tells the model when to use tools.
- Defines memory helpers and shared constants used in later cells.


In [25]:
# Orchestrator setup

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.tools import render_text_description
from langchain_core.runnables import RunnableLambda
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

rendered_tools = render_text_description(tools)

system_prompt = f"""You are an assistant with access to the following tools:

{rendered_tools}

Given the user query, return one of the following:
- a single JSON object with 'name' and 'arguments' keys
- a list of JSON objects if both tools are needed
- or {{"name": "no_tool", "arguments": {{}}}} if no tool is needed

Rules:
- Use apply_filters for explicit constraints like genre, actor, director, year range, or rating.
- For direct recommendation requests by genre such as 'good sci-fi movies', 'best comedy movies', or 'suggest horror films', you should use apply_filters.
- Use semantic_search for fuzzy requests like 'like Interstellar', mood, tone, or themes.
- Use no_tool for greetings, simple conversational replies, or movie questions you can answer directly without retrieval.
- If both are useful, return apply_filters first and semantic_search second.
- Parse the user's natural language into clean tool arguments before calling a tool.
- Normalize genre wording to the dataset labels when possible. For example, map 'scifi', 'sci fi', or 'science fiction' to 'Sci-Fi'.
- Available genre labels include Action, Adventure, Animation, Biography, Comedy, Crime, Documentary, Drama, Family, Fantasy, History, Horror, Musical, Mystery, Reality-TV, Romance, and Sci-Fi.
- For apply_filters, pass 'query_struct' as a JSON string with keys genre, actor, director, year_min, year_max, rating_min, semantic_query.
- If a field is missing in query_struct, set it to null.
- For semantic_search, pass 'query_text' and 'top_k'.
"""

session_memory = {}
MAX_HISTORY_TURNS = 5


# Create or return the message history for a specific chat session.
def get_memory(session_id):
    if session_id not in session_memory:
        session_memory[session_id] = ChatMessageHistory()
    return session_memory[session_id]


# Convert recent chat messages into plain text that can be passed to the model.
def format_chat_history(history):
    if not history:
        return "No previous conversation."

    return "\n".join(
        f"{message.type.title()}: {message.content}"
        for message in history[-(MAX_HISTORY_TURNS * 2):]
    )


# Clear the stored conversation history for one session.
def clear_chat_history(session_id):
    get_memory(session_id).clear()


prompt = ChatPromptTemplate.from_messages([
    ("system", "{system_prompt}"),
    ("user", "Conversation history:\n{history}\n\nCurrent user query: {input}")
])

rank_results = lambda results: sorted(
    [
        {
            **item,
            "score": round(
                (0.7 * float(item.get("similarity_score", 0.0) or 0.0))
                + (0.3 * max(0.0, min(float(item.get("imdb_rating", 0.0) or 0.0) / 10.0, 1.0))),
                4,
            ),
        }
        for item in results
    ],
    key=lambda item: item["score"],
    reverse=True,
)

KNOWN_GENRES = [
    "Action", "Adventure", "Animation", "Biography", "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy", "History", "Horror", "Musical", "Mystery",
    "Reality-TV", "Romance", "Sci-Fi"
]
GENRE_PATTERNS = {
    "Sci-Fi": ["sci-fi", "sci fi", "scifi", "science fiction"],
    "Reality-TV": ["reality tv", "reality-tv"],
}


### Routing Helpers
- These helpers clean raw model output and turn it into a usable tool plan.
- They also build fallback plans when the model output is missing or invalid.
- This makes the orchestrator more robust during real user interaction.


In [26]:
# Tool plan helpers

# Safely extract JSON content from raw model output or stringified payloads.
def extract_json_payload(value):
    if isinstance(value, (dict, list)):
        return value

    text = str(value or "").strip()
    if not text:
        return None

    fenced_match = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL)
    candidate = fenced_match.group(1).strip() if fenced_match else text

    try:
        return json.loads(candidate)
    except Exception:
        pass

    start = min([idx for idx in [candidate.find("["), candidate.find("{")] if idx != -1], default=-1)
    if start != -1:
        for end_char in ["]", "}"]:
            end = candidate.rfind(end_char)
            if end > start:
                snippet = candidate[start:end + 1]
                try:
                    return json.loads(snippet)
                except Exception:
                    continue

    return None


# Infer a basic structured query from natural language when fallback routing is needed.
def infer_query_struct(user_query):
    query = str(user_query or "")
    lowered = query.lower()
    genre = None

    for known_genre in KNOWN_GENRES:
        if known_genre.lower() in lowered:
            genre = known_genre
            break

    if genre is None:
        for known_genre, variants in GENRE_PATTERNS.items():
            if any(variant in lowered for variant in variants):
                genre = known_genre
                break

    year_values = [int(year) for year in re.findall(r"\b(19\d{2}|20\d{2})\b", query)]
    year_min = None
    year_max = None
    if len(year_values) >= 2:
        year_min, year_max = min(year_values[:2]), max(year_values[:2])
    elif year_values:
        if re.search(r"(?:after|since|newer than|from)\s*{}".format(year_values[0]), lowered):
            year_min = year_values[0]
        elif re.search(r"(?:before|until|older than|up to)\s*{}".format(year_values[0]), lowered):
            year_max = year_values[0]

    rating_min = None
    rating_match = re.search(r"(?:rating|rated|imdb rating)?\s*(?:above|over|greater than|at least|>=)\s*(\d+(?:\.\d+)?)", lowered)
    if rating_match:
        rating_min = float(rating_match.group(1))

    return {
        "genre": genre,
        "actor": None,
        "director": None,
        "year_min": year_min,
        "year_max": year_max,
        "rating_min": rating_min,
        "semantic_query": query,
    }


# Build a backup tool plan when the model does not return a clean routing decision.
def build_fallback_plan(user_query):
    query = str(user_query or "")
    lowered = query.lower()
    query_struct = infer_query_struct(query)

    if lowered.strip() in {"h", "hi", "hello", "hey", "help", "thanks", "thank you"}:
        return []

    has_explicit_filters = any(
        query_struct[key] is not None
        for key in ["genre", "actor", "director", "year_min", "year_max", "rating_min"]
    )
    is_fuzzy = any(token in lowered for token in ["like ", "similar", "mind bending", "mind-bending", "feel", "mood", "theme"])

    plan = []
    if has_explicit_filters or any(word in lowered for word in ["recommend", "suggest", "movie", "movies", "film", "films"]):
        plan.append({
            "name": "apply_filters",
            "arguments": {"query_struct": json.dumps(query_struct)},
        })

    if is_fuzzy or not plan:
        plan.append({
            "name": "semantic_search",
            "arguments": {"query_text": query, "top_k": 10},
        })

    return plan


# Normalize the model output into a consistent tool-plan format.
def normalize_tool_plan(model_output, user_query):
    parsed_output = extract_json_payload(model_output)
    if parsed_output is None:
        return build_fallback_plan(user_query)

    if isinstance(parsed_output, dict):
        parsed_output = [parsed_output]
    elif not isinstance(parsed_output, list):
        return build_fallback_plan(user_query)

    normalized_steps = []
    for step in parsed_output:
        step = extract_json_payload(step)
        if not isinstance(step, dict):
            continue

        name = step.get("name") or step.get("tool")
        if isinstance(name, str) and name.strip().lower() in {"no_tool", "none", "direct_answer"}:
            return []
        arguments = step.get("arguments", {})
        if isinstance(arguments, str):
            parsed_arguments = extract_json_payload(arguments)
            arguments = parsed_arguments if isinstance(parsed_arguments, dict) else {"query_text": arguments}
        elif not isinstance(arguments, dict):
            arguments = {}

        if not name:
            continue

        normalized_steps.append({"name": name, "arguments": arguments})

    return normalized_steps or build_fallback_plan(user_query)


# Remove explicitly mentioned source movies from similarity-style recommendation results.
def remove_reference_movies(user_query, ranked_results):
    comparison_words = [" like ", " similar to ", " similar ", " movies like ", " same as "]
    query_lower = f" {user_query.lower()} "
    if not any(word in query_lower for word in comparison_words):
        return ranked_results

    referenced_titles = {
        str(title).strip().lower()
        for title in df["Title"].dropna().unique()
        if str(title).strip() and str(title).strip().lower() in query_lower
    }

    if not referenced_titles:
        return ranked_results

    filtered_results = [
        item for item in ranked_results
        if str(item.get("title", "")).strip().lower() not in referenced_titles
    ]
    return filtered_results if filtered_results else ranked_results


### Execution Helpers
- These functions run the selected tools and combine their outputs.
- They rank the final movie list and remove repeated or reference titles when needed.
- They also generate the final conversational response for the user.


In [27]:
# Tool execution and response generation

# Run the selected tools, merge their outputs, and rank the final movie candidates.
def tool_chain(model_output, user_query, already_normalized=False):
    tool_map = {tool.name: tool for tool in tools}
    normalized_plan = model_output if already_normalized else normalize_tool_plan(model_output, user_query)

    merged_results = {}

    for step in normalized_plan:
        tool_name = step.get("name")
        if tool_name not in tool_map:
            continue

        chosen_tool = tool_map[tool_name]
        arguments = step.get("arguments", {}) if isinstance(step, dict) else {}
        if isinstance(arguments, str):
            parsed_arguments = extract_json_payload(arguments)
            arguments = parsed_arguments if isinstance(parsed_arguments, dict) else {}
        elif not isinstance(arguments, dict):
            arguments = {}

        if tool_name == "apply_filters" and "query_struct" not in arguments:
            arguments = {"query_struct": json.dumps({
                "genre": arguments.get("genre"),
                "actor": arguments.get("actor"),
                "director": arguments.get("director"),
                "year_min": arguments.get("year_min"),
                "year_max": arguments.get("year_max"),
                "rating_min": arguments.get("rating_min"),
                "semantic_query": arguments.get("semantic_query", user_query),
            })}

        if tool_name == "semantic_search" and "query_text" not in arguments:
            arguments = {
                "query_text": arguments.get("semantic_query") or user_query,
                "top_k": arguments.get("top_k", 10),
            }

        tool_result = chosen_tool.invoke(arguments)
        parsed_result = extract_json_payload(tool_result)
        if isinstance(parsed_result, dict):
            parsed_result = [parsed_result]
        if not isinstance(parsed_result, list):
            continue

        for item in parsed_result:
            if not isinstance(item, dict) or "row_index" not in item:
                continue
            row_index = item["row_index"]
            if row_index in merged_results:
                merged_results[row_index].update(item)
            else:
                merged_results[row_index] = item

    ranked_results = rank_results(list(merged_results.values()))
    ranked_results = remove_reference_movies(user_query, ranked_results)
    return ranked_results[:5]


# Generate the final conversational reply using chat history and ranked movie results.
def generate_final_response(user_query, ranked_results, history_text, used_tools=True):
    if not ranked_results:
        response_prompt = f"""
You are a movie assistant.

Recent conversation history:
{history_text}

The user asked:
{user_query}

Answer naturally and directly.
Treat short follow-up questions as referring to the movie or topic from the recent conversation history when that context is available.
If the user asks who the hero, lead, protagonist, or main character is, identify that character for the movie being discussed.
If there is no single clear hero, say that naturally, for example that the film is female-centric, ensemble-led, or does not revolve around one hero.
Do not give a generic definition of words like hero or protagonist unless the user explicitly asks for a definition.
If the question can be answered from general movie knowledge, answer it without mentioning tools, retrieval, or missing results.
If the user seems to want recommendations or filtered results and you do not have enough grounded movie matches, say that naturally and invite them to narrow or rephrase the request.
Keep the tone conversational and user-facing.
"""
        return llm.invoke(response_prompt).content

    response_prompt = f"""
You are a movie assistant.

Recent conversation history:
{history_text}


The user asked:
{user_query}

Answer naturally as a helpful movie assistant speaking directly to the user.
Do not mention internal terms like 'ranked candidates', 'results', 'filters', 'semantic search', or anything about how the system works.
Treat short follow-up questions as referring to the movie or topic from the recent conversation history when that context is available.
If the user asks who the hero, lead, protagonist, or main character is, identify that character for the movie being discussed.
If there is no single clear hero, say that naturally, for example that the film is female-centric, ensemble-led, or does not revolve around one hero.
Do not give a generic definition of words like hero or protagonist unless the user explicitly asks for a definition.
Use the movie list below as guidance, but present it in normal conversational language.
If the user asks for a factual detail about a movie, person, cast, director, rating, or year, answer that directly and briefly first.
Do not add extra movie recommendations, comparisons, or unrelated suggestions unless the user asked for them.
If the movie list contains relevant titles, recommend or discuss those titles directly instead of saying you do not have enough options.
Do not introduce movie titles that are not present in the movie list below.
If the available movie options are weak or limited, say that naturally without referring to the underlying process.

Movie list:
{json.dumps(ranked_results, indent=2)}
"""
    return llm.invoke(response_prompt).content


# Coordinate routing, tool execution, ranking, and final response generation for one user turn.
def run_movie_chain(inputs):
    user_query = inputs["input"]
    history_text = format_chat_history(inputs.get("chat_history", []))
    try:
        model_output = (prompt | llm | JsonOutputParser()).invoke({"system_prompt": system_prompt, "input": user_query, "history": history_text})
    except Exception:
        raw_response = (prompt | llm).invoke({"system_prompt": system_prompt, "input": user_query, "history": history_text})
        model_output = getattr(raw_response, "content", raw_response)
    normalized_plan = normalize_tool_plan(model_output, user_query)
    ranked_results = tool_chain(normalized_plan, user_query, already_normalized=True)
    final_response = generate_final_response(user_query, ranked_results, history_text, used_tools=bool(normalized_plan))
    return {
        "results": ranked_results,
        "tool_plan": normalized_plan,
        "output": final_response,
        "response": final_response,
    }


### Session Wrapper
- This wrapper connects the orchestrator to session memory.
- It allows the chatbot to remember earlier turns in the same conversation.


In [28]:
# Wrap the orchestrator with session memory

movie_chain = RunnableWithMessageHistory(
    RunnableLambda(run_movie_chain),
    lambda session_id: get_memory(session_id),
    input_messages_key="input",
    history_messages_key="chat_history",
)


## Test Movie Chain
- Runs a sample query through the orchestrator.
- Useful for checking whether routing, retrieval, and response generation are working.


In [29]:
print(movie_chain.invoke(
    {"input": "Suggest some good action movies?"},
    config={"configurable": {"session_id": "demo-session2"}},
)["response"])

Here are some great action movies you might enjoy:

1. **The Dark Knight** (2008) - Directed by Christopher Nolan, this film has an impressive IMDb rating of 9.0. It's a gripping tale in the Batman saga, focusing on the conflict between Batman and the Joker, showcasing thrilling action and deep themes.

2. **The Lord of the Rings: The Return of the King** (2003) - Directed by Peter Jackson, this epic conclusion to the Lord of the Rings trilogy also boasts a 9.0 IMDb rating. It features a large ensemble cast as they confront the ultimate battle for Middle-earth.

3. **The Lord of the Rings: The Fellowship of the Ring** (2001) - Also directed by Peter Jackson, this film has an IMDb rating of 8.9. It sets the stage for the epic adventure, introducing key characters like Frodo, Aragorn, and Gandalf.

Each of these films delivers intense action, memorable characters, and captivating stories! If you’d like to know more about any of these, feel free to ask.


## Follow-Up Example
- Demonstrates how the same session remembers earlier context.
- The second question depends on the first one, which helps verify memory handling.


In [30]:
demo_session_id = "demo-followup-session"

print("First question:")
print(movie_chain.invoke(
    {"input": "Tell me about Interstellar"},
    config={"configurable": {"session_id": demo_session_id}},
)["response"])

print("\nFollow-up question:")
print(movie_chain.invoke(
    {"input": "Who is the hero?"},
    config={"configurable": {"session_id": demo_session_id}},
)["response"])


First question:
"Interstellar" is a 2014 sci-fi film directed by Christopher Nolan. The story is set in a near-future where Earth is facing ecological collapse. It follows a group of astronauts led by Cooper, played by Matthew McConaughey, who are on a mission to explore potential new habitable planets through a wormhole near Saturn. Cooper's primary motivation is to find a new home for humanity, as well as to save his children, especially his daughter Murph, who plays a significant role in the film as well.

The movie intricately weaves themes of love, sacrifice, and the survival of the human species with stunning visuals and a thought-provoking narrative. It also features a strong ensemble cast, including Anne Hathaway, Jessica Chastain, and Michael Caine, among others. Would you like to know more about a specific aspect of the film?

Follow-up question:
In "Interstellar," the main protagonist is Joseph Cooper, played by Matthew McConaughey. He is a former pilot who takes on the miss

## Handle Edge Cases
- Handles empty or vague input more gracefully.
- Detects follow-ups without context and conflicting filters.
- Catches reset commands, fallback text, and unexpected runtime errors.
- Keeps these replies more conversational for the user.


In [31]:
# Check the edge cases and handle them appropriately

MOVIE_KEYWORDS = {
    "movie", "movies", "film", "films", "actor", "actors", "director", "genre", "rating",
    "imdb", "watch", "recommend", "recommendation", "cinema", "plot", "cast", "sequel"
}

INTERNAL_FALLBACK_MESSAGES = {
    "No matching movies found for your question.",
    "No movies matched your current filters. Try relaxing constraints like genre, year, actor, director, or IMDb rating",
    "No movies matched your current filters. Try relaxing constraints like genre, year, actor, director, or IMDb rating.",
}


# Detect impossible or contradictory rating and year conditions in the user query.
def has_conflicting_filters(user_input):
    query = user_input.lower()

    above_matches = [float(x) for x in re.findall(r"rating(?: above| over| greater than| >=)\s*(\d+(?:\.\d+)?)", query)]
    below_matches = [float(x) for x in re.findall(r"rating(?: below| under| less than| <=)\s*(\d+(?:\.\d+)?)", query)]
    if above_matches and below_matches and max(above_matches) > min(below_matches):
        return True

    after_years = [int(x) for x in re.findall(r"after\s*(\d{4})", query)]
    before_years = [int(x) for x in re.findall(r"before\s*(\d{4})", query)]
    if after_years and before_years and max(after_years) > min(before_years):
        return True

    year_ranges = re.findall(r"from\s*(\d{4})\s*(?:to|-|until|through)\s*(\d{4})", query)
    if any(int(start) > int(end) for start, end in year_ranges):
        return True

    return False


# Check whether the query looks like a follow-up that depends on prior context.
def looks_like_context_dependent_followup(user_input):
    query = user_input.lower()
    markers = [
        "better than", "worse than", "more like that", "same as that", "that one",
        " it ", " this ", " that ", " them ", " those ", " these "
    ]
    padded_query = f" {query} "
    return any(marker in padded_query for marker in markers)


# Check whether the query appears to be about movies using keywords or known titles.
def is_movie_related_query(user_input):
    query = user_input.lower().strip()
    if any(keyword in query for keyword in MOVIE_KEYWORDS):
        return True

    titles = [str(title).strip().lower() for title in df["Title"].dropna().unique()]
    return any(title and title in query for title in titles)


# Handle edge cases and safely route a user message through the movie chatbot.
def safe_movie_chat(user_input, session_id="default-session"):
    cleaned_input = (user_input or "").strip()

    if not cleaned_input:
        return "Tell me what kind of movie help you want, and I’ll jump in."

    if cleaned_input.lower() in {"h", "hi", "hello", "hey"}:
        return "Hi! I can help with movie recommendations, details, comparisons, genres, cast, and ratings. What would you like to know?"

    if cleaned_input.lower() == "help":
        return "You can ask me things like 'Recommend sci-fi movies like Interstellar', 'Who directed Inception?', or 'Show action movies after 2015 with high ratings'."

    if cleaned_input in INTERNAL_FALLBACK_MESSAGES:
        return "That looks like one of my internal fallback messages rather than a real movie question. Ask me directly about a movie, genre, actor, director, rating, or recommendation and I’ll help."

    if len(cleaned_input.split()) < 2 and not is_movie_related_query(cleaned_input):
        return "Could you tell me a bit more? I’m here to help with movie-related questions."

    if cleaned_input.lower() in {"reset", "clear", "clear chat", "new chat"}:
        clear_chat_history(session_id)
        return "All set — I cleared the chat history for this session."

    if has_conflicting_filters(cleaned_input):
        return "I’m seeing conflicting filters in that request. Could you adjust the rating or year conditions and try again?"

    memory = get_memory(session_id)
    if looks_like_context_dependent_followup(cleaned_input) and len(memory.messages) == 0:
        return "That sounds like a follow-up, but I don’t have earlier context in this session yet. Could you mention the movie or recommendation you mean?"

    if not is_movie_related_query(cleaned_input) and len(memory.messages) == 0:
        return "I’m focused on movie-related questions right now. Ask me about movies, genres, actors, directors, ratings, or recommendations and I’ll help."

    try:
        result = movie_chain.invoke(
            {"input": cleaned_input},
            config={"configurable": {"session_id": session_id}},
        )

        if not isinstance(result, dict):
            return "Something went wrong while I was working on that. Could you try asking it again?"

        response = result.get("response") or result.get("output")
        if response:
            if response.strip() == "No matching movies found for your question.":
                return "I couldn’t find a good movie match for that exact request. If you want, try loosening the genre, year, actor, director, or rating constraints a little."
            return response

        return "I couldn’t put together a useful answer for that one. Try rephrasing it and I’ll take another shot."

    except KeyError as exc:
        return f"I ran into a missing field while processing that request: {exc}."
    except json.JSONDecodeError:
        return "I had trouble understanding the tool plan for that request. Try rephrasing it a bit and I’ll try again."
    except Exception as exc:
        return f"Something went wrong on my side: {type(exc).__name__}: {exc}"


# Example edge cases
print("Example 1: Empty input")
print(safe_movie_chat("", session_id="edge-demo"))
print("Example 2: Greeting")
print(safe_movie_chat("H", session_id="edge-demo"))
print("Example 3: Clear chat command")
print(safe_movie_chat("clear chat", session_id="edge-demo"))
print("Example 4: Very vague non-movie query")
print(safe_movie_chat("weather", session_id="edge-demo"))
print("Example 5: Conflicting rating filters")
print(safe_movie_chat("Show sci-fi movies with rating above 9 and rating below 5", session_id="edge-demo"))
print("Example 6: Follow-up without prior context")
print(safe_movie_chat("Is it better than that one?", session_id="fresh-session"))
print("Example 7: Impossible year range")
print(safe_movie_chat("Recommend horror movies from 1800 to 1700", session_id="edge-demo"))


Example 1: Empty input
Tell me what kind of movie help you want, and I’ll jump in.
Example 2: Greeting
Hi! I can help with movie recommendations, details, comparisons, genres, cast, and ratings. What would you like to know?
Example 3: Clear chat command
All set — I cleared the chat history for this session.
Example 4: Very vague non-movie query
Could you tell me a bit more? I’m here to help with movie-related questions.
Example 5: Conflicting rating filters
I’m seeing conflicting filters in that request. Could you adjust the rating or year conditions and try again?
Example 6: Follow-up without prior context
That sounds like a follow-up, but I don’t have earlier context in this session yet. Could you mention the movie or recommendation you mean?
Example 7: Impossible year range
I’m seeing conflicting filters in that request. Could you adjust the rating or year conditions and try again?


## Create Final UI
- Builds a Gradio chat interface for the movie assistant.
- Uses session IDs so users can continue follow-up conversations.
- Connects the UI directly to the edge-case-safe chatbot flow.


In [32]:
# Create a UI using gradio or any other tool of your choice

import gradio as gr


# Process one UI message, update chat history, and return the refreshed interface state.
def chat_with_movie_bot(user_input, session_id, history):
    session_id = (session_id or "default-session").strip() or "default-session"
    history = history or []

    if not (user_input or "").strip():
        return history, "", f"Using session: {session_id}"

    response = safe_movie_chat(user_input, session_id=session_id)
    history.append({"role": "user", "content": user_input})
    history.append({"role": "assistant", "content": response})
    return history, "", f"Using session: {session_id}"


# Reset the UI conversation and clear stored memory for the selected session.
def clear_ui_chat(session_id):
    session_id = (session_id or "default-session").strip() or "default-session"
    clear_chat_history(session_id)
    return [], "", f"Chat history cleared for session: {session_id}"


with gr.Blocks() as movie_app:
    gr.Markdown("# IMDB Movie Chatbot")
    gr.Markdown(
        "Ask for recommendations, movie details, comparisons, ratings, or follow-up questions. "
        "Use the same session ID to keep conversation memory."
    )

    with gr.Row():
        with gr.Column(scale=1):
            session_id_box = gr.Textbox(
                value="demo-session",
                label="Session ID",
                placeholder="Enter a session name...",
            )
            status_box = gr.Textbox(
                value="Using session: demo-session",
                label="Status",
                interactive=False,
            )
            gr.Markdown(
                "### Try asking\n"
                "- Recommend good sci-fi movies\n"
                "- Tell me about Interstellar\n"
                "- Who directed Inception?\n"
                "- Show action movies after 2015\n"
                "- Which one is better?"
            )

        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Conversation", height=500)
            user_input_box = gr.Textbox(
                label="Your message",
                placeholder="Ask about movies, recommendations, ratings, genres, or follow-up questions...",
            )
            gr.Examples(
                examples=[
                    "Recommend mind bending sci-fi movies like Interstellar with rating above 8",
                    "Suggest some good comedy movies",
                    "Tell me about Interstellar",
                    "Who directed The Dark Knight?",
                ],
                inputs=user_input_box,
                label="Quick examples",
            )

            with gr.Row():
                send_button = gr.Button("Send", variant="primary")
                clear_button = gr.Button("Clear Session")

    send_button.click(
        chat_with_movie_bot,
        inputs=[user_input_box, session_id_box, chatbot],
        outputs=[chatbot, user_input_box, status_box],
    )
    user_input_box.submit(
        chat_with_movie_bot,
        inputs=[user_input_box, session_id_box, chatbot],
        outputs=[chatbot, user_input_box, status_box],
    )
    clear_button.click(
        clear_ui_chat,
        inputs=[session_id_box],
        outputs=[chatbot, user_input_box, status_box],
    )


movie_app.launch(debug=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


## Final Summary
- This notebook builds a movie chatbot using the IMDb dataset, LangChain, OpenAI, FAISS, and Gradio.
- The workflow begins with loading the dataset, understanding the available movie fields, and creating a richer `movie_description` column from title, year, genre, rating, director, cast, and duration.
- It then prepares the data for retrieval by splitting the movie descriptions into chunks, generating embeddings, and storing them in a FAISS vector database.
- An initial RAG pipeline is created to show how the chatbot can retrieve relevant movie context and answer questions from that retrieved information.
- After that, the notebook moves to a tool-based chatbot design with two tools:
- `apply_filters` performs exact filtering using dataset columns such as genre, actor, director, year range, and IMDb rating.
- `semantic_search` performs fuzzy search for requests based on meaning, tone, theme, or similarity to another movie.
- The orchestrator sits on top of these tools and controls the chatbot flow.
- It asks the LLM whether a tool is needed, normalizes the tool plan, supports a `no_tool` path for direct answers, runs the selected tools, merges and ranks results, and then generates a user-friendly response.
- The orchestrator also uses session-based memory so that follow-up questions can refer back to the current conversation.
- Follow-up behavior is improved so short context-dependent questions like asking about the hero or main character can use the recent chat history.
- Edge-case handling is included to make the assistant more robust.
- It covers empty input, vague questions, non-movie questions, follow-ups without context, conflicting filters, reset commands, internal fallback messages, and unexpected runtime errors.       s
- The notebook includes test cells for direct movie queries and follow-up queries so the chain can be checked before launching the interface.
- Finally, the chatbot is exposed through a Gradio UI, where users can ask questions, keep a session ID for memory, and clear the session when needed.

## Future Enhancements
- Add web search support to fetch up-to-date movie information beyond the local dataset.
- Add poster images and richer movie cards in the UI for a better browsing experience.
- Add filters for language, certificate, duration, or metascore to support more specific searches.
